# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdPYtW40h2v1JLZxZ71riBfmTWG3ZDg6ebDI8ONt3Zg4mObJeNFlny6gHtJfx77qNKqpJKxvRMcnLimdPYVj1u3Xvrvqv8uDWJo1kw7y5XWz2xFSyWcZKJOB1F6m0ii7d5HkyLD1mwKJ+kt3kWhOWz20T60yCaj6JZEi9EtlrCB6GeXiyzII78sCNOgzTriONgAv8eRivVeulnt2Ew1s0/w8dRNIpeiWOZyUkm+tF9kMTRQkZZT3yM43koxVEc+mNxn4rTeOIDICcD7+ji9PCDOIC1dHHErvwGs6Wt0dZrWHEGnUdbbeFHUxHFWb3RUW+Er602Tj2VM8BDGof30pvGD1EY+1OZeNMgabXFzp9FmiW9USTgNdqi/+j98SryFwGAE65ECGBlMtXgHifBvRTlUAKGgqXFyaoj0nyJ60aEpbfwZpJnaYdH7OovdjI/mcss3RmvdoIpw5YjVju0ovTWT+RUzOIQxk7Fw20Ac85zP/Fh4RJHnsE3KQ8KTcV5/0v/UjwkQQaIUcAA1FlsA5zEcdatLzSYEQ41zhUq8DX2U4loMsjgj1P829Kf4SmgSbY8D2HyvHa77J7ILE+iouvf4iBq6SE7AIJBizTz58hwTDDs/ErsdcXnJFj4yUpMbuXkrteIwJa1TN1IqEYijpjBFGjqaw/RASsrOer1FPu/bpjFQFeF3cwB2wb6smRlfMLXLE5g501FQFgJoTcyYVP/YhwPZzOIQJg0e3Vw1Ha9pyKt7hekNB0N55oIX4iLIMqlA4zacvQLcJcEsD0OxLXshvGDxI2Fa5XVlfLUN+5hAFo/WrXusJMeEkehL66BX/zMH20h5yT+g3ojZ0EkgTY3bQEtsbvsymiaPgTApKOtbrBcRWMUFQU4auSm9eNLMS5D626W5uMaQai5zdjE0A1rrXARjMgCzaYWfv08qNCq3kZ+m8glCFz6A9JlY4o/03Hpp2m5Tfe74gi3p5j4URyhwMR30wCoJQvBGGhxVTwibinHre3CsxXt5tcmLjsvai92xEDt5PVdByRv6X1qzcd9FLciA02QgcolGHip03NSJZt7/4CqDQ1Owo8kYCfu7UzNTS5Rs+H3Lvah7xsIb8CsxToPgzIDt1z1aRPSgcs7taGqjaHRzfOMjADYjTbgYmbIhqUxHmrUMOaclMz8pisG0k8mt8K9UuICkDJKOYPp4LOuR3kjtkvm2eYRFyvq7lIzBe6a1Irq+7xGQYm2qkhaR+eiSx6G3rIqvVSHDo/mZj56pAV8l9auBa0l8hrI3KxAmBjujcDgNohRBdhvuSuMYf/v7I4SSQ27RL82lPn4qu8ax9IZX4oGz29fbvidmuRtsflYIrM9p2BMDSHt2k6mEG/eU+Yo37exmkYo+jl3l9nrN9liDqZeT6QGAukXGBB1qLnxBhaN05oBWNz2zLNAUpvvZKJjOfPzMBMzcNzGvmWWlNac8gana+SywxAwetUlAz9qF4xXNP6fEBXORhUR8vI16lUqEcPd8evji6/npxeHx/1L7/jkEgZs8qZHEay84kieXhwdnnpfLy5/GXw+POrb4GRyscSpZZhK1f7D4aCvpnmp01mfymJnPTIydBFEwA8MRZvDFIMsySeAAsCc5VYem6bs5eHXCow0g42owlOBkS/7P5+c94837KS9GqPj2cWXk/7A1d8YGjsv4nsE0ew7/LJBv+zeS2VS7Xox/FRQfV3vOLtVsuH4cHi42TLZn4MuZ/3hIXX7t8HFea2fGo/WJjMfe3X/lsZRQbH+8lYuZAI7nMJH4suZTSxmiwaSVXhG0cwLeHjVlda6aW8kno5lDC8PB5+8wfDw48n5xw0HyBI/vbUiItyoCbGOIQrM8rO1+LUHd2N52D/7rKauzAb0vjz5onFj01iF/UAfJME4z3BDfb4SR2GeZmA6/x7cxjy6S2HD0Uptmq1BWZ2RTFwdfbo6/2WwDuXGI5IEBIWBLTXCZpg2eh+dXg2G7s1iYnfC67e64JD9S+ecxqjY+yFO7mRiT3h2SH9+PjmtibxK74WPU5d0VQtuBrzOHCb4dvc1i6jNU12K3WDNgpwjVZdVZTvNckcUK88TH60IxSre8dXl4fDk4twb9I9gtv23u93dwrYAo1QsgijHeMUSRriTq1kCqkekco4RbNGKwSRZgNAZ+6EfTaSIZ4INkCwOQSbhV78XYM8pLfLz1aAvTl5faE71TvuolYYnZ/2Lq6GC4Y8ahFfwFqYCZTVNRSj9VIppAT7j2/vUP7wcfugfDiuDvH0Hg7wSZ35yJxjVANssBMXCYUFxC7Z2NpZ+Rrbu23cpI+6LTMYxTHQaz2lb/h5MqnE+F+dxFszAjsLJUzGEp6GE3d//cPXRO734OOiJcRyjB4dqWnKQvzuXGZgm/fPDD6cgI4q2rHb30I79HdoDu0S2wV+BqGfYgEjfw8D8up2UroCqCzCW5yXVQWxhiJnSDmnP1uA6v0DowKVN4uVKC7B/v+pfAYgf1k3491zmsjsdGyzb1K2+cazOCOuJCkt3hB8++KtU5AB3SAAOBsdi8O+nQSYp1I+RfBkGwIgYU7PWRPyEbXagJy1q5gchWC/pnwTp5IcARsWR7WW6LDVjLfbiLPvMaFUdkpRxlMLsIpITmaYYuzdCgIKcg1GE/MaGsUq1VCS6UtXwxjQy6paQCurZZk7ZsLBdOqIkhCX968oCvjNli0O2dZy2ZkdYVkbxsbAcOqJmC+hGJXBVBVR8Y8LUKHVHkbb/bXcWGHPh30kgRNqadpgIXnx3MExyqSznNS6Wdq9eofic5Ekio8kKHPWFv7yNgc9AQHxl8XIKHJpB27PD//A0STV8wC1v+MnpxVdv8LmPVK232UWZtwusMQ0443Z1fnpydjLsH2OYmmfPUJzupEsJkl07IGpWhe5yvD3K/6lGYN8qoOFJkeTsFktpuQAHBMF0Hk3nOUc6j0GkAtj4tyfyCDcqap2JgS7k+BTBBiUB2zkJJnehNKGHxvegDAH1G0FpL1QJlGGwkHGekdM9IPR89bPJ7TSei8ucEobF4oALL6sKA7WO0aSk0/DTZX/w6eL02PvwmZC6uyt+hH/333aJXvj5lw+vU3dna469fZrEQjTDctkfXsK2Rg2MY55EQRaAJPwDfASPVKWX3oAkzDDUl6HLlqVakO4s5CIGWTP00zsxTHwlCQHpGCnbwWS3CFFcLmIYF2QR5rYzaExZiBv13kMRaiEd1MUderX42JvEeYRWBHApz/sxjMdKm8DX6FEmcZga/MZPPH+ScQwYN9woYu9uo4eNAHFSO/URS/+Q5AaTR4z/kN50JrYHqn0qdAfOaiRxmu4sQz+DTwtYyAL0WDAOwiBbETPZ6VV/JrNV15FBpslr6d/R1n0wlbFuS7NiBKGb5uNWsn2NefrXP/6lN9r6lz//1812h2DtULu2u8tolP6B2omiYReNvWWrbYUw8Ml1D1juBiGEBRN62uLPyIYCVRo10dgcr1DUeex+TWUoM9mi+AJqdQ82JsVOSuyiueOoG/hAw1SLBYY4qvgQaMl6FmMcExQ0K3iqnLA6BBEofXweap9WFm4ucke69CfYOUkzpZGgcYRG6gJETIT5/wQ2Bc4SZDxIuOqK4W0Almwi76FJasH29xy2FnJxmi9IBzSXCtSxgslXRyFGvWG7ziA/+0CJ5/SWQ3m69Bi+8igAW8tTXDPTDvUj1r908Z+3rXb3Vn677v108+Q9apDRbKQ95AD6SSMAX4RLYJA0qxp9DiANYAwQXxH90R63iR4j0ZngVOJBBm7Zjet0ukhVB5QdAzJrritQSdEd8xgPPg3Su3W5S2OgSrAVGjJbtZyTKZKyLHMZFsIHK9C0LkAWZy0g0jVxIW+eG1BcCcZQemBz5CHX+PD2FERMEluPdRQ89cSjfLJC1PXIfn2569m0vnRH+8bkYokKFzqcoezKCGp/sJCagBcY5UuelWo/PJBH/opUAJh3xde9olDrGhrcKDul7RBYg4kfpfVKIdFyB425lIImFeDO4M4nz9sqSppMQOaD2sTiKV2XhN6IBDKWBTpl4ZMI0jQHUQWDkUczv41he4F/BnpfiTeqjALdYwg2GHEBvg3qeofwdsuvEkMwJ9tuuorMUQHFJKjmco0xAOANMrsO0ehI0OrJXLKwLLFxJHmfteHtkV+hdwyWTJQapJxqzxmxoSOAJk2ZzZa6p1W9AaoZrQZo3EpGW//Jru6PoxHwR/fHf0ItDS1OPp5fXPaPDgf9dqexaxF7xN7IVuv63xhpH4QW9Foqk3sKUYixnKENrTZMTzc8mRXOINjfgFGkza0PdNUOOtLUCoHi7Okkl6SOxyj3S52NTOCiF8pFb1qrEWoMCOh+bmZRo1WlEm3/+iRFKr9hDjVPNVfOg9k5O6wwA4OxfPhnsesWjYWmLpfYpKD1S6kzjMLMiP48SUcjz9GlVBdHTNcbcUm0mYp7PwymJRGJOoY0A8X6qMZ9MsoMC1ohxV+WetiAZNjJSTTXbAbZmidbQzuarJl6/Ph/nH44TadE6ffQ0Fp9MyGxWanrNzc0ilmVkcHVpaXDSFudBF9pTfA4JEsoo7rLX1hFAkpEG3Ny/txRu9lYbFMr2CTEFuNUealeuocQQUuExxbYdSMIngDVs8lti6FwFVa5SIUJRqxuJrVBJMEaqBk4sogvHOqpxrL4cvpXxcrq7TW2/0BhnNpo4BLfFQpWt7XZ2wH+kWqYL8Wj6vSkNeD4dWnHODwyLp/W0dvLPOLFKwWDVcaLeJpDc1X5jkUAeUQcFqhQryvq+oxFB9OB+tsK4/lcJo1l/uApuEv7sU7R+mJ9cX+lpL9TGJEqwkSdd9AMA08CeSBO/DlmOFT0YxR58H09bnGpAxdA+HE+98DTGodyWuQMUMKgdu7z92glTnUhB+4xP8UvoCemJDpY0wGKGEQBJiGm6EfByImcAPfh/LBxcSHXuA60fmktNzcq1uNxFPKj9+lkMLy4/CvGlXZ3FeSFae+2n4vIi8x4JS1rJbZ5zYtBcwL8LXp7r3Ir1BUxSBkWnwIwAoxlxUGpFV2Zc5jJxhw/oqrDAuMG59uNAXiETsNq7LUyVXPtTNPcULnG3mgLubkyKAVQKHez1ttixuJjKjXphUdXyvlgMtcq8bVhDRDSJkgVadxRmkuyhVNmOViVmybgGygYLFool6wKZckUmodaDh6yuUM/An8mxVMKPsZuUmBbjHCTgUmaA9kfWVq0ZHfeRSljCqW2k1MsRn6OU6zGQIGlOrCDK/IwIOhnHkqQNPMXy+qBGYUPLWW6xZsofqD6tWSGH1ujrR8+9X446/0wMM7keCj6Eg9DTjHMj6nJlo2jr8ESRDHW2y5lhIJEjEZ4gmKCAVcWX2CSg37D1FgoKLlJ8hSQqnbVxUDoyKa9rWxW9VCmeeiqIrAwC7D8HzjEKH4Uf3xHH+jrzTMmtEZj3Ex+yzh8WInKklNLbVI0b8C5i3MQwH6exYB9dQYpkbCOCeIgj2Q0iackUa6iAN7Cqm/9hE66hHLuT1ZCIXXdkkFndHmqbglf2/l8FubpbcteeUtN3Y/o3ySJQc5eDOjN+tJKgB74DIxTME2SVjkNWr60Mqrl6HDAgj3sPJvt/GQKGgIQUYtAo6qBP13qLEHMTcCyQjjSA6zHQbxh1Vx3KvXzirlRQ0UxdHNDCycvFFGhvJehl64WY5DK9MEVtA/vUTXS426+XMpEz4bRbHyGsvnk/OeL0ZYr7D7K9/fe/HGUz+TuTKNOhmZfkrxNnac/vZnCv9Pdaa0zmDOwSb4eXp5zIh/fnZx/tCu/TTje+7vr4OhfXl5cNi7in99O3N0GV0dH/cGguePuu4rcpq939/e52pH2p1wEJLUNMnRE5s/VuwVmsedSfQIpMgFBiWZlYbpQeM4VVzvC8IPWJzgN1qjwo8MlyrNU+Q6w5YMihTXOZzMJU5XiAA0t0gcBqRg+I0gw0GO10x3RrkaZr8P45WpoTMKAxRlFaKxitlU8Wiu6hK8M40MO5WHHs1bQ6NEearRVtAayiuLoZPmcgMRn9Kb2GChHHf157ZEiJT5Wb+u90VLG7qjCSHEZID8Z2sKwNbs+EbPlcOUw5WQ2Rfe7an1WcGmNvIyXrV0rVAHr8rgqBr2aR/j4dMPGGbzVFpkpJ5kGGdmA3CV9uhGPdQHUfhKPanR4pxD0VJYF4+sVuU8JqJl5rJUx6VwVayNWwgeRfCC2pTSTgTaXwnfr4BJuVrxRJWaD5qhtr9t4dJ81MeMctWJia7j28+EP2l7xEmlsdUWJSOcUtS47KPRXG6MSs4bi91l33dLNlhsfiWIZF2Hl1qrlFmq25PoUzG93gIxxgrnfyCj5Egt0RzOMl7dUvRKRW+L55bmfTOFpitVvbEeza2abp6Ws1WqLJG0BjyVfNdaVaZ1PsJ5oozUMuO2G0DeDWKiXjaFkt2MTGI/B+AnQiZoG/jyKUzDyaMO0NE5jNGEDjc3SG2kGV4nsdcCyqtLQPvhJtBGwOk5Gu9lG68bI1MbChpgk620j4Mje/FWgKetjY9jAcDWldIvqfXqYsOfaGLLiXPEHpxvKI7G0VDkekBFpeQhhCoPYZnyjXldWDqaHLL1zvcMw3jStgUxd90KqNSQa7lJKLfIwC0gNYBEG8Ek8c6xqsxWUaS4n/DUTph5zNfAw2jpn6ymySldh4DiZAuArifG8siuuQRcEFUK5cmy8qmZsfSyvt+Gb7RutlPVnl2YuZtQGRDGCtoBwnIqqhsf0efvG1tfwvWJc6GTpi9LyRUXC4Weatm2mlBNmZXtrkbJPDdsUcXkbYNBvtRk5LWuG5tHxzFsZgkOj6pxeHtakvIjzYpPKXSZ5EobBuJvIv+cSS0/tr0nMlF3z8TKJUXdsGC11xEn5VpRRCZwOfKmPHMc1t+F8mXtBNIs5tGKLDIscJ1G6lCALBJeFCPD4pw94/Qfy6PmXk+OTQzpgga3s7YbD27b2aAunpeonwDyQGvpxFWflqPx94i+8xRhb7VoPKFydcG89z5MrzIAViQcGbruUbvHg62Wetez9cD3aiu4D0Ik76SJg33JnByiXrHYA3gOEt8O82KVCwA5D4anSSd2DZdPBJL3vRPGtxPNp8CaPsDp166Zi9aPoO7CrNCrlZPha4kFMlA1L/YxTLXQ6NM+66TIMsNS9g6fwak4A9Qbr/0C8qYZHgTTXJjVuKAQHza93b5xNC4pgS8xvcOu9m2qWkJtrOpXj7t9sGMEyxAcOZsRXI1C3E8+/BxsGrZKGSCtdC5EiDs6/9M+PxK28n7y+3X//li1jvNsG2VfdmENGD2w4sHWUVGgOXa3nKVj1bLZYyrniBz0b0r4kdwfjbCAADoxhjvtfzq9OT13iE4HndaNoJ6JzaAoXVHnwLH6rlT2TEEubPpFM5G3oiCcc5mC/xos4T2mbX1KlaKKLryOMApb5pjSAj766WImFLapBCUS6U6GC4yCdxLhzxJkPcN8Hvr556JiTkwBVsEgxVZbFYt8swKaKTsFnn9R2YiSnNEwhmYjuHLBIlN2AxR6Y+pvqIzSpdWGRCYEjpIHO89HF+dHV5WX/fKgq5bFmWPuGFNMBeQrWnddKZTjrKG7ygun6kih8YYeuFsgcozTks9EOK5jwCXKh2cfeyl0VewS5wPKZGRJDkeWToUxDv3ygpQvx1jmg1LQZaK5iPTB5+R7az/SZKu+xAFAfJX9qqH+sjY58ZNROMk+K1qO9zG29SjA32rUx+DyWB0q2KYtmNVdcayUDK024StpLwzhLe6wkQfSVardqauNMj7s9mq0j9vjN05pBm0utrS5c9IjF55V7sciu0iFt1vlI0eppSSJz5fRzu0PipbCHLPPNjFg4IKhHKcptMFX72yvJQUOQsLbIYuwAbeT701RwPzGW/kSlxTnnovaomYeyZDXJa3X3llUmhq86jhyn/hBPFmZ1qtB1ELF+ArDdeW7OxvOGTTOvOQjpmL+cvuJKUGGGws2zZanuy8AaL1Ep41GUSUm+IwRFrONTaRLV3uEphtas4e4V4CeshcRCHT7mV/LZumuvsBvVcgJB4H3lkovbLFs23yCDL6U7oWs3YWHJdx/Ue2x4F4qz6FYn+wkjEi8mQwNyIr0s5koNpVwA+CxP9VHF0VYwDaUNv4qhcDh/EoPfgabqziyRUh8dMXeYQxmu3WiKXLU4eqEZ0ES39UYt7F3I/KJt8U2trek3rFF9tX6GI1HtVlq0tV6MXupE7+rpAOA5rzjJWond11onco4JdkCqV2vsDPR/hxgzD+S9VIyVR6JfKsI2mPUZEWbM3Sy+UOk1SrDnY+6ZU2VVezxQLL12vR71NW0ci6mf3LWSNF4hGHlkWu13SEcSidN8sWzhpgNYOnhQERT5wf73B+jxPRmtKPk8FadQwkVG02VMuUaybHDWXqPJo81ZS7u7I5D4UrFySkP6kfg0HH4Wny8GQzQkP/aHQkHCyBsn8UMKgop9aUo7+vdxgCfr43w6C9HWH+MZDXUIuSalVLiuah06w3a2hchaBsj+WOn89KjRYxmxdSaElXhKStpD40uvqCZBGUdXsOqdwzndFIORjrP4H0EY+q/xbH3ra4BH/VJxPhR7u93dPwn44v3bP4lv79+2McUbyq9y/EuQvX735p+7b96L1i+fhmenHREGd1J8BIc1boujW7Co5Gs6F4n/iYE/85NAdbGCMSVUhxPkK4bIh3lUaPM17wBnl//YYT9x5+S4rhDsDk/2R0xCIPYCLpWnU671GQwkF9skpX3S1mUZ5Sard1dkAGVwxLbxznC1lLr+rL7GGoWhnR3q616qrQRf89Y50DB29HQH6m8FIGL5ymjwkSQIfO6Q1gA3/2DvHcmLRKZLB0r4EiiMF6Hmwrz6/q6r+Bpf43i64nuVlnjz3bRVFKuswxqvnTZNYa6lLRyrXpZiRjy7uNs5cVKpklZwo2oA39/Dc6+jLX3jXSGO6pCo8lRKKYHT2D/93Cc9hB4kSpZHPOUxlU8Y4in3LdYL43IB9nrJsFMauBZzdXn6v7UWYM4IrLPiwNJvtJzGonUDHM4tVuApO9YA2RCCBnNXG0oe7gXlQJbOoxHpw5f2GbmLDt+VliztpyLGRPxw+PmEnACurP5A9m9NaQAQ3tJfIVf/v7FwnVamIrHK01doXFRiTeSe0EgurhwltE7iOJni5SBxEakx9GS72+1WUkKpjlvZVgc4Uv4yeK3MwteaCfj8VUELu8YloQJNusRcuYHaZm9zKVFpc1uFWvgCEyiY84GKsrv+UuG5UyFX27k7dH1ABXeqFABPGaxECUgD6mDzbT/q6Z+2f2chzbgBpTJ3TVCMtvgqK2Z1npYv7dFlrH8S6QqcySltiMrl5eTyjfWOsIpeOGpV90QJ4QeF81nb3RxQK3d3ClCUTpMyNhd4y8Ik3czAdEoANibpnoVlEs8TLMJA1igvGgK7EbFe2+nlLldQoBX6+FRrcG3t+hvNxQ5DZlMeN1zHjqgz+HqMq/YVvu+UYQB7q7ibq21SFr6+QP6XsuGPb8QnvZQe36wmXj2aM25TuB6Q9G27/QRiwnq2nGTbHQH2Z7vX3Zs9/dAW/yrsFrNlii2gK7xzMRmdMHhWGsR3Bk9z9lkrZ0dw8jn35XMchip9ge5phLXARVaK8xNKyGCmocZ2WmGyg2HxRQnWXwruOqi6nLayamA4PclzYvPWT5WNUo1+baoe3oE7BlDjhWeK/np0m/bxDORc8SgLslACabf/LR7D023RGtIdK5zdAVOibInfe/w9tGw3WRiJfD6eptNAJeU74m/xmLI0FNkvIFafwd7mqdXpHK7HxZo7POIIyrUnZsCtSEtk42YhlY/xliLFHAoMFM2I3oncQET9KkOE14gN+V2tQbFubFN8qDWz0YFt7W9qHUxUYXPrs8so2VSCWpR0S9HNRJh8q+6YAwau6Wbg2Yli6gIpTyb5WsC+8JCx+tSuWjwvEFE+edfq+tWSY9mLZRQrhsUV9yolGx285ynTfIzc2MyL5ef+NzmhSwZ16nQHocALBDFTyulzsCGilA574DGP4p44PH5boKdjqi4+MbMEaxE0y6RUw2mH74rh6xUEXlea3sI03cXdvRV/NrOwLqjxolQ+tsi0OSCMXPMNqiZ3GmYvsqmzT5Wjb0zyzbzJbK7aKqrpe1Y9VWED7PT4ZNCcOaEcX+88Y9yCk8pWxva7qdQbOw55VxDQILi5mLHC99cDYBPxqJjl6UaQD9sTgzhPJlp3FRcU0HxI92O+g+ixMnOjt1eUGNhrVgne2t031QWZexjzjvC8Gpd1ZiVhgUFEmdVihd5jObUNLnfmmo6NhkZWeWZsO249oyMrxRI65ZRVkjkuh3lZFs64JebXpqTKZ/RjSEe40ZEA+u7R+hHl8vYgcP9QOhjioHpY36Z0pyRw2555vys+5EE4FT//jDU1hhgC6bsASWrMASPQZWR8TZratlrKGg/ZSI7ALbMYYZLMat3gO+/eD7HHGzPMfu8hDJNaeypN4Wc8y6fu/vt34rX41P9yZN/bkHpUjIQyH/5y9Y4elsovDrGH8W3ZF401qvnBGixHPVRFdKip6rxWjONghHvPT+acetramfTueTlmuRFVN9GlJ+oi8+Vb9eXk79wek6yAweovLjmc2eYZw2D8jfDDIyczc+AaCPcyWSH3WYLWMd/3Ld4owvrfXPy771m8wdeLKY9rlaVh2f800EWLK/WXyh6LvWiXElr7iy7O5S1UvRViMQXhlWFNMa7lnmE2+pqlglZja5gfGSM1S3LHrycZ8FtMCHmRnPtUVZOlujLiTnr/kEns6qPDBIqAwVL29mrtClldfn1TkVFvuuLUBxf9VgspVdDHgR6KSJRBCHUH5Mbht/33/p6oq2vlrxe+lmGWgjVmWaMUnW0uoKrG5xB4u7zxM+UegFQddTjYLFr8fPK5v6aY0Sp5HOcztPkP9iwEchJ9TCf10BRyJcWx9M9bTpR/Vfl+tkyd3/MlrQdUlGsfJXP/4ArfTh0RBtQ5W1eAHlsd0J96oa5+YeUtNzCqSwDOg+/9fSJjoeRotnh4VfeLw17v3Wh4GpI1v/rXefgAbG1dhOc1K7OIsQ7ujWdEEwxZxMvXYTSKH5r4yXzBDC1sumNxItVLY2J1DZZoU1XCmY5Ervl61gevd9jIJ693g91CHdTOebY9BdZ6BaM9254IWvSgT8/3KQpqzJBjc6+nBpLha71b7xabDmmplQB4NXrpve7ujAKN4r/4JmP9jBbpvl6ogMoWZsBXdluSLQ8+cP+as/uuRBx1vAvCsMrGL3PziimmOaVxNBXWJu1qbhyGkxEefk7m+O/w+mz3Paylu4MNqteDGU8PDmr3g71odTUfYcbHHFsACJ6FmaLvWgG8OZBYWzX9Rhj5QWUZufKUG3701gg8rDn7WwlKrDn8W3WjCtx1qpEN2wlVkTbDw9XYr3YDV6WFd2urK7brYbSGXNcmW60I9bQeNUQU7xdnH9pdMQCriE9h6FLDipn1rssRulUtPlfJmNjRXSVlDclZxVWnQFBlxvddvrHLiHdhJCBO8JcFjMspCccvcvTdyn5D91/xU4Oj/8LqL4Phq4k6fQ84ETWM46WuOjVCjI5aU86/cyk7/2KlwEtSYU/Q+TDkk9bC/yb26T3/sgk8adfLtqbgM+B1O2l55x2++CepreL9mnf3fL0XkQFtgzSUctl647oDrrjgrkI9vOhdRaeNDFL9okQK0Tro3LCwYoHoN7gPCTSwQ63xtSLSjQou1rs9l84t9bPr8kXsbMWkcZKCNQ7U36ZyyN9ubWZm59dNg/EAP1qVV+LjzzmnZuEZ31dLlxtWB+7e+2GO9+W5fQG+5dUcf+1aX5Bj1y+Dk3e776oiwR1zMNjQfcshxo4MVoU2qMCNr34Q+7sura1f62y0msY4gXF74ivYRvo2zDKZah75qjqsDhS8sXInICQ4u1uTVaRIVMGQpCRIQNmNeAmGHHr0CIl19gxFmOrAhK8KrWZ3ntwd0Kvvf6pr1Weyq/KnXXWYDudXp6LweBb+fBkeXuwpD78oVKnGuNfMAs+VzD451uOYWeYXjAPQ9dbFGuonuVRtEj47+yC+XB6etV80IyenPukjf4c6b4WKTzxu/1Wm2yoEWj81Ssfit89j0cIDyD+rn+xsb79szcbv0ui1O88JPin2BWWIdxb4kYzzNFxp9feiOfk3sU7jyu9i4Yr5GsVj0VK/pNWm9asBjcsJefHHJwPV/ExfUZLS7R+bY8Fi64oltokkMw0u4mZ7w5mbrcBVGcghK8N1bQLZGbC7wReY83EZN1lqlpl1DI+vHG3xiYMDTkdXraKOwEDlQQu/6LSx2hcMs8h1vCDjKEbVUFCL0DcxZBZSnjV2KDujT9MKukN4Bg6to8Ls6vL0O+2kSgtt+7jO+LkV4PNTrGcyLQwn8O9sb7dYr07WH5XL1JvQqEFs9NgZra5qU2csyqlFDcAdtXijfP/N7I2l1Wp0UQew4H/HkSu0lYvTfbViytpi1uy4B4YBL0ZTq2wcylClVVtCv5ps41ficy3lz9dK+ngrZmZ5xfh6maUGEuw3NdDcIbyGQJXSJAoAdasKC7JqpMrA4N5ucdkJ7GBPnbVdABeoK3HV9XZ8dbBxCrl6ny/eWccVY7hOZQWwWVAerjfMEe2xV24LNqZTQCsZe2DdAaC3gO2qqssR0KSyL+T8Ra7GMajhE/yVqiRfmqHzdcqjtHGme3pNyhwJ9FB8JXOewhhicJtntJXwd66q20FBt2YTqB+hVJoH73Ej28TzuPrGI7p4XpHVqlIML5B5+m99rVea"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
